# KarmaDock 虚拟筛选教程

**论文**: Zhang et al., *KarmaDock: a deep learning paradigm for ultra-large library docking with fast modular confidence estimation*, 2023. 
**核心创新**: 采用 **E(n)-等变图神经网络 (EGNN)** 直接预测配体原子坐标，实现分子对接。

---

### 核心思想

KarmaDock 将蛋白-配体对接建模为一个"去噪"问题：
1. 将蛋白口袋原子与（加噪的）配体原子构建交互图；
2. 通过 **E(n)-等变消息传递**：特征更新保持旋转不变性，坐标更新保持等变性；
3. 通过多次 **recycle**（循环精炼）逐步修正配体坐标。

### E(n)-等变性的关键

EGNN 的坐标更新公式只依赖原子间 **方向向量** 和 **标量权重**，因此对输入施加任意旋转/平移后输出会做相同的旋转/平移——这就是 E(n)-等变性。

### 与打分模型的核心区别

| 对比项 | 打分模型 | 对接模型 (KarmaDock) |
|--------|----------|---------------------|
| 预测目标 | 标量 (logKa) | 坐标 (N x 3) |
| 损失函数 | MSE | RMSD |
| 训练方式 | 回归 | 对配体加噪声，学习恢复正确构象（去噪） |

---

### 学习目标

1. 理解 E(n)-等变图神经网络的坐标更新机制
2. 掌握"加噪-去噪"的对接训练范式
3. 理解 Recycle 循环精炼机制（参考 AlphaFold2）
4. 学会用 RMSD 评估对接模型的预测质量

In [ ]:
# ============================================================
# 环境设置、路径配置与依赖导入
# ============================================================

def find_project_root(marker='demo_data'):
    """向上查找包含 demo_data/ 的项目根目录"""
    from pathlib import Path
    here = Path('.').resolve()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f'无法找到包含 {marker}/ 的项目根目录')

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'demo_data'
CORESET_FILE = DATA_DIR / 'CoreSet.dat'
COMPLEX_DIR = DATA_DIR / 'coreset'

import os
import warnings
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from rdkit import Chem
from rdkit import RDLogger
import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline

# 抑制 RDKit 的冗余警告信息
RDLogger.logger().setLevel(RDLogger.ERROR)
warnings.filterwarnings('ignore')

# 版本信息
version_info = pd.DataFrame({
    '库': ['PyTorch', 'NumPy', 'RDKit', 'pandas'],
    '版本': [torch.__version__, np.__version__, Chem.rdBase.rdkitVersion, pd.__version__]
})
display(version_info)

print(f"\n项目根目录: {PROJECT_ROOT}")
print(f"数据目录:   {DATA_DIR}")
print(f"CoreSet:    {CORESET_FILE}")
print(f"复合物目录: {COMPLEX_DIR}")

## 1. 超参数设置

KarmaDock 的关键超参数包括：

- **DISTANCE_CUTOFF**: 交互图边的距离阈值，控制蛋白-配体原子对的连接范围
- **N_EGNN_LAYERS**: 每轮 recycle 内的 EGNN 层数，决定消息传递的深度
- **N_RECYCLES**: Recycle 次数，多次循环实现"粗调 -> 细调"
- **NOISE_TR_SIGMA**: 配体平移噪声标准差，模拟初始构象的不确定性

In [ ]:
# ============================================================
# 超参数定义
# ============================================================

DISTANCE_CUTOFF = 8.0       # 交互图边的距离阈值 (Angstrom)
ATOM_FEAT_DIM = 10          # 简化原子特征维度
HIDDEN_DIM = 128            # 隐层维度
N_EGNN_LAYERS = 4           # 每轮 recycle 内的 EGNN 层数
N_RECYCLES = 2              # Recycle 次数（重复 EGNN 传递）
NOISE_TR_SIGMA = 5.0        # 配体平移噪声标准差 (Angstrom)
N_EPOCHS = 200              # 训练轮数
LR = 1e-3                   # 学习率
BATCH_SIZE = 1              # 变长图，逐样本处理
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

# 展示超参数汇总
hp_df = pd.DataFrame({
    '超参数': ['距离阈值', '原子特征维度', '隐层维度', 'EGNN层数', 'Recycle次数',
              '噪声标准差', '训练轮数', '学习率', '批大小', '随机种子', '设备'],
    '值': [f'{DISTANCE_CUTOFF} A', ATOM_FEAT_DIM, HIDDEN_DIM, N_EGNN_LAYERS, N_RECYCLES,
           f'{NOISE_TR_SIGMA} A', N_EPOCHS, LR, BATCH_SIZE, SEED, str(DEVICE)],
    '说明': ['蛋白-配体原子对建边的距离上限', '元素 one-hot (9) + 芳香性 (1)',
             'EGNN 隐层特征维度', '每轮 recycle 内的 EGNN 消息传递层数',
             '循环精炼次数，共享 EGNN 参数', '配体平移噪声 sigma',
             '训练迭代次数', 'Adam 学习率', '变长图逐样本处理',
             '可复现性', '自动检测 GPU']
})
display(hp_df)

print(f"\n总 EGNN 传递次数: {N_EGNN_LAYERS} layers x {N_RECYCLES} recycles = {N_EGNN_LAYERS * N_RECYCLES} passes")

## 2. 数据加载与特征提取

数据来源为 PDBbind CASF-2016 核心集（285 个蛋白-配体复合物）。

**特征提取流程：**
1. 解析 `CoreSet.dat` 获取 PDB ID 与结合亲和力标签
2. 用 RDKit 加载蛋白口袋 (`.pdb`) 和配体 (`.mol2` / `.sdf`)
3. 为每个原子构建 10 维特征向量：元素 one-hot (9维) + 芳香性 (1维)

**对接模型的加噪策略：**
- 对配体坐标施加随机旋转（绕质心）和随机平移
- 模拟"未知初始构象"，让模型学习从噪声中恢复正确位姿

In [ ]:
# ============================================================
# 特征提取函数
# ============================================================

# ---- 元素符号 -> one-hot 映射 (9 类 + 1 芳香性 = 10 维) ----
ELEMENT_LIST = ['C', 'N', 'O', 'S', 'F', 'P', 'Cl', 'Br']  # 8 种常见元素 + 1 other


def parse_coreset(path):
    """解析 CoreSet.dat，返回 {pdbid: logKa} 字典。跳过以 '#' 开头的注释行。"""
    labels = {}
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            pdbid = parts[0]
            logKa = float(parts[3])
            labels[pdbid] = logKa
    print(f"[INFO] CoreSet.dat: {len(labels)} complexes")
    return labels


def atom_features(atom):
    """
    构建 10 维原子特征向量：
      - 前 9 维: 元素类型 one-hot (C, N, O, S, F, P, Cl, Br, other)
      - 第 10 维: 是否为芳香性原子
    """
    feat = np.zeros(ATOM_FEAT_DIM, dtype=np.float32)
    symbol = atom.GetSymbol()
    if symbol in ELEMENT_LIST:
        feat[ELEMENT_LIST.index(symbol)] = 1.0
    else:
        feat[8] = 1.0       # other 类别
    feat[9] = float(atom.GetIsAromatic())
    return feat


def load_mol(path, fmt):
    """
    用 RDKit 加载分子文件。
    fmt: 'mol2', 'sdf', 'pdb'
    先尝试正常加载，再尝试 sanitize=False 后手动 sanitize。
    """
    mol = None
    if fmt == 'mol2':
        mol = Chem.MolFromMol2File(path, sanitize=True)
        if mol is None:
            mol = Chem.MolFromMol2File(path, sanitize=False)
            if mol is not None:
                try:
                    Chem.SanitizeMol(mol)
                except Exception:
                    pass  # 保留 unsanitized 的分子
    elif fmt == 'sdf':
        supplier = Chem.SDMolSupplier(path, sanitize=True)
        for m in supplier:
            if m is not None:
                mol = m
                break
        if mol is None:
            supplier = Chem.SDMolSupplier(path, sanitize=False)
            for m in supplier:
                if m is not None:
                    mol = m
                    try:
                        Chem.SanitizeMol(mol)
                    except Exception:
                        pass
                    break
    elif fmt == 'pdb':
        mol = Chem.MolFromPDBFile(path, removeHs=False, sanitize=True)
        if mol is None:
            mol = Chem.MolFromPDBFile(path, removeHs=False, sanitize=False)
            if mol is not None:
                try:
                    Chem.SanitizeMol(mol)
                except Exception:
                    pass
    return mol


def random_rotation_matrix():
    """
    生成一个随机 3D 旋转矩阵（均匀采样 SO(3)）。
    使用 QR 分解方法：对随机高斯矩阵做 QR 分解，Q 即为均匀旋转矩阵。
    """
    M = np.random.randn(3, 3)
    Q, R = np.linalg.qr(M)
    # 确保行列式为 +1（避免反射）
    Q = Q @ np.diag(np.sign(np.diag(R)))
    if np.linalg.det(Q) < 0:
        Q[:, 0] *= -1
    return Q.astype(np.float32)


def compute_rmsd(pred, true):
    """
    计算两组坐标的 RMSD (Root Mean Square Deviation)。
    pred, true: (N, 3) numpy array 或 torch tensor
    RMSD = sqrt( mean( ||pred_i - true_i||^2 ) )
    """
    if isinstance(pred, torch.Tensor):
        diff = pred - true
        return torch.sqrt((diff ** 2).sum(dim=-1).mean())
    else:
        diff = pred - true
        return np.sqrt((diff ** 2).sum(axis=-1).mean())


def add_noise_to_ligand(coords):
    """
    对配体坐标施加随机噪声，模拟"未知初始构象"：
      1. 绕质心施加随机旋转（改变朝向）
      2. 施加随机平移（改变位置，sigma=5A）
    返回加噪后的坐标副本。
    """
    coords = coords.copy()
    center = coords.mean(axis=0)

    # ---- 随机旋转（绕质心）----
    coords_centered = coords - center
    R = random_rotation_matrix()
    coords_rotated = coords_centered @ R.T
    coords = coords_rotated + center

    # ---- 随机平移 ----
    translation = np.random.randn(3).astype(np.float32) * NOISE_TR_SIGMA
    coords = coords + translation

    return coords


def load_complex_data(pdbid):
    """
    为单个蛋白-配体复合物加载坐标和特征。

    返回:
      prot_coords : (N_p, 3)  蛋白口袋原子坐标
      prot_feats  : (N_p, 10) 蛋白口袋原子特征
      lig_coords  : (N_l, 3)  配体真实坐标（晶体构象）
      lig_feats   : (N_l, 10) 配体原子特征
    如果解析失败则返回 None。
    """
    pocket_path = os.path.join(str(COMPLEX_DIR), pdbid, f"{pdbid}_pocket.pdb")
    ligand_mol2 = os.path.join(str(COMPLEX_DIR), pdbid, f"{pdbid}_ligand.mol2")
    ligand_sdf = os.path.join(str(COMPLEX_DIR), pdbid, f"{pdbid}_ligand.sdf")

    # ---- 1. 加载蛋白口袋 ----
    prot_mol = Chem.MolFromPDBFile(pocket_path, removeHs=True, sanitize=False)
    if prot_mol is None:
        return None
    try:
        Chem.SanitizeMol(prot_mol)
    except Exception:
        pass

    # ---- 2. 加载配体 (mol2 优先, sdf 备选) ----
    lig_mol = Chem.MolFromMol2File(ligand_mol2, sanitize=False)
    if lig_mol is None and os.path.exists(ligand_sdf):
        lig_mol = load_mol(ligand_sdf, 'sdf')
    if lig_mol is None:
        return None
    try:
        Chem.SanitizeMol(lig_mol)
    except Exception:
        pass

    # ---- 3. 去氢 ----
    try:
        prot_mol = Chem.RemoveHs(prot_mol)
    except Exception:
        pass
    try:
        lig_mol = Chem.RemoveHs(lig_mol)
    except Exception:
        pass

    # ---- 4. 提取 3D 坐标与原子特征 ----
    prot_conf = prot_mol.GetConformer()
    lig_conf = lig_mol.GetConformer()

    prot_coords = np.array([prot_conf.GetAtomPosition(i)
                            for i in range(prot_mol.GetNumAtoms())], dtype=np.float32)
    lig_coords = np.array([lig_conf.GetAtomPosition(i)
                           for i in range(lig_mol.GetNumAtoms())], dtype=np.float32)

    prot_feats = np.array([atom_features(a) for a in prot_mol.GetAtoms()], dtype=np.float32)
    lig_feats = np.array([atom_features(a) for a in lig_mol.GetAtoms()], dtype=np.float32)

    if prot_feats.shape[0] == 0 or lig_feats.shape[0] == 0:
        return None

    return prot_coords, prot_feats, lig_coords, lig_feats

In [ ]:
# ============================================================
# 加载数据并展示一个样本
# ============================================================

labels = parse_coreset(str(CORESET_FILE))

all_data = []
failed = 0
for pdbid in sorted(labels.keys()):
    result = load_complex_data(pdbid)
    if result is None:
        failed += 1
        continue
    all_data.append(result)

print(f"[INFO] 成功加载 {len(all_data)} 个复合物, {failed} 个失败")

# 展示第一个样本的形状
sample = all_data[0]
sample_df = pd.DataFrame({
    '数据': ['蛋白坐标', '蛋白特征', '配体坐标', '配体特征'],
    '形状': [str(sample[0].shape), str(sample[1].shape),
             str(sample[2].shape), str(sample[3].shape)],
    '说明': [f'{sample[0].shape[0]} 个蛋白原子 x 3D 坐标',
             f'{sample[1].shape[0]} 个蛋白原子 x {sample[1].shape[1]} 维特征',
             f'{sample[2].shape[0]} 个配体原子 x 3D 坐标',
             f'{sample[3].shape[0]} 个配体原子 x {sample[3].shape[1]} 维特征']
})
print("\n第一个样本的数据形状：")
display(sample_df)

## 3. 数据集与数据加载器

对接模型数据集的核心设计：

- **标签不是标量**：与打分模型不同，对接模型的标签是配体的真实 3D 坐标 (N_l x 3 矩阵)
- **动态加噪**：每次 `__getitem__` 时对配体坐标施加不同的随机噪声（旋转 + 平移），相当于数据增强
- **BATCH_SIZE=1**：由于每个复合物的原子数不同（变长图），使用逐样本处理

In [ ]:
# ============================================================
# Dataset 类、DataLoader 构建
# ============================================================

class DockingDataset(Dataset):
    """
    对接数据集。每个样本包含蛋白口袋和配体的坐标与特征。

    与打分模型数据集的关键区别：
      - 打分模型的标签是标量 (logKa)
      - 对接模型的标签是配体真实坐标 (N_l x 3 矩阵)
      - 每次取数据时，对配体施加随机噪声（旋转+平移），模拟"未知初始位姿"
    """
    def __init__(self, data_list, add_noise=True):
        self.data = data_list
        self.add_noise = add_noise

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        prot_coords, prot_feats, lig_coords_true, lig_feats = self.data[idx]

        # 动态加噪——每次取数据时噪声不同，相当于数据增强
        if self.add_noise:
            lig_coords_noisy = add_noise_to_ligand(lig_coords_true)
        else:
            lig_coords_noisy = lig_coords_true.copy()

        return (torch.FloatTensor(prot_coords),       # (N_p, 3)
                torch.FloatTensor(prot_feats),         # (N_p, 10)
                torch.FloatTensor(lig_coords_noisy),   # (N_l, 3)  加噪坐标
                torch.FloatTensor(lig_coords_true),    # (N_l, 3)  真实坐标（标签）
                torch.FloatTensor(lig_feats))          # (N_l, 10)


# ---- 随机 80/20 划分训练/测试集 ----
indices = np.random.permutation(len(all_data))
split = int(0.8 * len(all_data))
train_idx, test_idx = indices[:split], indices[split:]

train_data = [all_data[i] for i in train_idx]
test_data = [all_data[i] for i in test_idx]

train_loader = DataLoader(DockingDataset(train_data, add_noise=True),
                          batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(DockingDataset(test_data, add_noise=True),
                         batch_size=BATCH_SIZE, shuffle=False)

# 展示数据划分结果
split_df = pd.DataFrame({
    '数据集': ['训练集', '测试集', '总计'],
    '样本数': [len(train_data), len(test_data), len(all_data)],
    '比例': [f'{len(train_data)/len(all_data)*100:.0f}%',
             f'{len(test_data)/len(all_data)*100:.0f}%', '100%']
})
display(split_df)

# 展示一个 batch 的数据形状
for prot_coords, prot_feats, lig_noisy, lig_true, lig_feats in train_loader:
    print(f"\n一个 batch 的数据形状：")
    shapes_df = pd.DataFrame({
        '张量': ['prot_coords', 'prot_feats', 'lig_noisy', 'lig_true', 'lig_feats'],
        '形状': [str(tuple(prot_coords.shape)), str(tuple(prot_feats.shape)),
                 str(tuple(lig_noisy.shape)), str(tuple(lig_true.shape)),
                 str(tuple(lig_feats.shape))],
        '说明': ['蛋白原子 3D 坐标', '蛋白原子特征', '加噪后的配体坐标（输入）',
                 '配体真实坐标（标签）', '配体原子特征']
    })
    display(shapes_df)
    break

## 4. 模型架构

KarmaDock 的核心是 **E(n)-等变图神经网络 (EGNN)**，包含两个主要组件：

### 4.1 EGNNLayer — 等变消息传递层

每层 EGNN 同时更新节点特征和坐标：

1. **消息计算**: $m_{ij} = \text{MLP}([h_i \| h_j \| \|x_i - x_j\|])$ — 消息只依赖距离（标量），天然旋转不变
2. **注意力权重**: $a_{ij} = \sigma(\text{Linear}(m_{ij}))$ — 学习边的重要性
3. **特征更新**: $h_i \leftarrow h_i + \sum_j a_{ij} \cdot \phi(m_{ij})$ — 标准图注意力聚合
4. **坐标更新**: $x_i \leftarrow x_i + \sum_j w_{ij} \cdot \frac{x_i - x_j}{\|x_i - x_j\| + \epsilon}$ — 仅更新配体原子

### 4.2 ToyKarmaDock — 完整模型

- 蛋白和配体各自嵌入后拼接成统一图
- 经过 N_RECYCLES 轮 x N_EGNN_LAYERS 层传递
- 每轮 recycle 重新初始化特征、重建边，但保留坐标更新

In [ ]:
# ============================================================
# EGNNLayer: E(n)-等变图神经网络层
# ============================================================

class EGNNLayer(nn.Module):
    """
    E(n)-等变图神经网络层 (Equivariant Graph Neural Network Layer)。

    EGNN 的核心创新在于坐标更新公式的设计：

      传统 GNN 只更新节点特征 h_i，不涉及坐标。EGNN 同时更新坐标 x_i，
      并且保证更新后的坐标满足 E(n)-等变性（旋转+平移等变）。

    具体步骤：
      1. 消息计算: m_ij = MLP([h_i || h_j || ||x_i - x_j||])
         消息只依赖节点特征和原子间距离（标量），不依赖方向，
         因此消息本身是旋转不变的。

      2. 注意力权重: a_ij = sigmoid(Linear(m_ij))
         学习每条边的重要性，用于加权聚合。

      3. 特征更新: h_i <- h_i + sum_j  a_ij * phi(m_ij)
         标准的图注意力聚合——特征是标量，天然旋转不变。

      4. 坐标更新 (仅配体原子):
         x_i <- x_i + sum_j  w_ij * (x_i - x_j) / (||x_i - x_j|| + eps)
         其中 w_ij = psi(m_ij) 是标量权重，(x_i - x_j) 是方向向量。

         这就是 E(n)-等变性的关键：
         - 对所有坐标做旋转 R：方向向量 R(x_i-x_j) = R * 原方向
         - 位移也跟着旋转 R -> 输出坐标也旋转 R -> 等变性成立！
         - 平移同理：全体平移后方向向量不变 -> 等变性成立

    参考 KarmaDock 原始实现 (EGNN_Block.py) 中的 coords_update 类。
    """
    def __init__(self, hidden_dim):
        super().__init__()
        # 消息 MLP: [h_i || h_j || distance] -> message
        self.msg_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU()
        )

        # 注意力: message -> 标量权重 in (0, 1)
        self.attn_fc = nn.Sequential(
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

        # 特征更新 MLP: message -> 特征增量
        self.feat_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU()
        )

        # 坐标更新权重: message -> 标量（控制坐标位移大小和方向）
        self.coord_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, h, x, edge_index, n_prot):
        """
        参数:
          h          : (N, hidden_dim) 所有原子的节点特征 [蛋白在前, 配体在后]
          x          : (N, 3)          所有原子的坐标
          edge_index : (2, E)          边索引（双向）
          n_prot     : int             蛋白原子数（仅更新索引 >= n_prot 的配体坐标）

        返回:
          h_new : (N, hidden_dim) 更新后的特征
          x_new : (N, 3)          更新后的坐标（蛋白坐标不变）
        """
        src, dst = edge_index[0], edge_index[1]

        # Step 1: 计算原子间方向向量与距离
        diff = x[src] - x[dst]                              # (E, 3) 方向向量
        dist = torch.norm(diff, dim=-1, keepdim=True)       # (E, 1) 标量距离

        # Step 2: 计算消息（旋转不变——只用距离，不用方向）
        msg_input = torch.cat([h[src], h[dst], dist], dim=-1)  # (E, hidden*2+1)
        msg = self.msg_mlp(msg_input)                           # (E, hidden)

        # Step 3: 注意力加权聚合 -> 特征更新
        attn = self.attn_fc(msg)                                # (E, 1)
        weighted_feat = attn * self.feat_mlp(msg)               # (E, hidden)
        # scatter add: 将每条边的消息聚合到源节点
        agg = torch.zeros_like(h)
        agg.index_add_(0, src, weighted_feat)
        h_new = h + agg                                         # 残差连接

        # Step 4: 坐标更新（仅配体原子——蛋白作为刚体不可移动）
        coord_w = self.coord_mlp(msg)                            # (E, 1) 标量权重
        direction = diff / (dist + 1e-6)                         # (E, 3) 归一化方向
        coord_shift = coord_w * direction                        # (E, 3) 加权位移
        # 聚合坐标位移到各节点
        coord_agg = torch.zeros_like(x)
        coord_agg.index_add_(0, src, coord_shift)

        # 蛋白原子坐标不变，仅更新配体原子
        x_new = x.clone()
        x_new[n_prot:] = x[n_prot:] + coord_agg[n_prot:]

        return h_new, x_new

In [ ]:
# ============================================================
# ToyKarmaDock: 完整模型
# ============================================================

class ToyKarmaDock(nn.Module):
    """
    KarmaDock 玩具模型：E(n)-等变 GNN 直接预测配体原子坐标。

    工作流程：
      1. 蛋白原子和配体原子各自通过嵌入层映射到隐空间
      2. 将蛋白 + 配体原子拼接成统一图（蛋白在前，配体在后）
      3. 经过 N_RECYCLES 轮 x N_EGNN_LAYERS 层的 EGNN 传递：
         - 每层同时更新特征（旋转不变）和配体坐标（旋转等变）
         - 每轮 recycle 重新初始化特征，但保留上一轮更新过的坐标
         - EGNN 层参数在不同 recycle 之间共享（减少参数量，防止过拟合）
      4. 最终的配体原子坐标即为预测结果

    Recycle 机制（循环精炼）——参考 AlphaFold2：
      直觉上，一次传递可能不足以将配体从随机位姿修正到结合位姿，
      多次 recycle 让模型能在逐渐修正的坐标上重新建边、重新传递，
      实现"粗调->细调"的效果。
    """
    def __init__(self, atom_dim=ATOM_FEAT_DIM, hidden_dim=HIDDEN_DIM,
                 n_layers=N_EGNN_LAYERS, n_recycles=N_RECYCLES):
        super().__init__()
        self.n_recycles = n_recycles

        # 蛋白原子嵌入层（将 10 维原子特征映射到 128 维隐空间）
        self.prot_embed = nn.Linear(atom_dim, hidden_dim)
        # 配体原子嵌入层（蛋白和配体使用不同嵌入，因为两者化学环境不同）
        self.lig_embed = nn.Linear(atom_dim, hidden_dim)

        # EGNN 层列表（参数在各轮 recycle 间共享）
        self.egnn_layers = nn.ModuleList([
            EGNNLayer(hidden_dim) for _ in range(n_layers)
        ])

    def forward(self, prot_feats, prot_coords, lig_feats, lig_coords):
        """
        参数:
          prot_feats  : (N_p, atom_dim) 蛋白原子特征
          prot_coords : (N_p, 3)        蛋白原子坐标（固定，不会被更新）
          lig_feats   : (N_l, atom_dim) 配体原子特征
          lig_coords  : (N_l, 3)        配体加噪坐标（初始值，将被 EGNN 修正）

        返回:
          lig_coords_pred : (N_l, 3) 预测的配体坐标
        """
        n_prot = prot_coords.shape[0]
        # Step 1: 嵌入——将原子特征映射到隐空间
        h_prot = self.prot_embed(prot_feats)     # (N_p, hidden)
        h_lig = self.lig_embed(lig_feats)        # (N_l, hidden)

        # Step 2: 拼接蛋白和配体坐标成统一图
        x = torch.cat([prot_coords, lig_coords], dim=0)   # (N_p+N_l, 3)

        # Step 3: 多轮 Recycle 循环精炼
        for _recycle in range(self.n_recycles):
            # 每轮重新初始化特征（只保留坐标的更新结果）
            # 这防止特征在多轮传递中不断累积误差
            h = torch.cat([h_prot, h_lig], dim=0)         # (N, hidden)

            # 基于当前坐标重建边（坐标变化后邻居关系可能改变）
            edge_index = self._build_edges(x, n_prot)

            if edge_index.shape[1] == 0:
                continue  # 极端情况：无邻居对，跳过本轮

            # 逐层 EGNN 传递
            for layer in self.egnn_layers:
                h, x = layer(h, x, edge_index, n_prot)

        # Step 4: 提取最终预测的配体坐标
        lig_coords_pred = x[n_prot:]               # (N_l, 3)
        return lig_coords_pred

    def _build_edges(self, x, n_prot):
        """
        基于距离阈值构建蛋白-配体交互边（双向）。

        为简化，只建立蛋白-配体之间的交互边，
        不建立蛋白-蛋白或配体-配体的内部边。
        这与 KarmaDock 关注蛋白-配体界面交互的设计一致。
        """
        prot_x = x[:n_prot]      # (N_p, 3)
        lig_x = x[n_prot:]       # (N_l, 3)

        # 计算蛋白-配体距离矩阵
        dist = torch.cdist(prot_x, lig_x)   # (N_p, N_l)

        # 筛选距离 < CUTOFF 的原子对
        prot_idx, lig_idx = torch.where(dist < DISTANCE_CUTOFF)

        if len(prot_idx) == 0:
            return torch.zeros((2, 0), dtype=torch.long, device=x.device)

        # 配体索引转为全局索引（蛋白在前，配体在后）
        lig_idx_global = lig_idx + n_prot

        # 双向边：蛋白->配体 和 配体->蛋白
        # 双向是必要的：消息需要双向流动
        src = torch.cat([prot_idx, lig_idx_global])
        dst = torch.cat([lig_idx_global, prot_idx])
        edge_index = torch.stack([src, dst], dim=0)   # (2, 2E)

        return edge_index


# ---- 实例化模型 ----
model = ToyKarmaDock().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# 展示模型架构和参数量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"模型参数总量: {total_params:,}")
print(f"可训练参数:   {trainable_params:,}")
print(f"\n模型结构：")
print(model)

## 5. 训练

训练流程：
1. 每个样本：对配体坐标加随机噪声（旋转 + 平移）
2. 模型从加噪坐标出发，通过 EGNN 预测去噪后的坐标
3. 损失：预测坐标与真实坐标的 RMSD
4. 梯度裁剪（max_norm=10.0）防止 EGNN 坐标更新导致的梯度爆炸

**注意**：若加噪后配体距蛋白过远导致无交互边，模型输出无梯度，需跳过该样本的反向传播。

In [ ]:
# ============================================================
# 训练循环
# ============================================================

print(f"训练 {N_EPOCHS} 轮...\n")

train_losses_history = []   # 记录每轮平均训练 RMSD

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    epoch_losses = []

    for prot_coords, prot_feats, lig_noisy, lig_true, lig_feats in train_loader:
        # 去掉 batch 维度（BATCH_SIZE=1 的变长图）
        prot_coords = prot_coords.squeeze(0).to(DEVICE)   # (N_p, 3)
        prot_feats = prot_feats.squeeze(0).to(DEVICE)     # (N_p, 10)
        lig_noisy = lig_noisy.squeeze(0).to(DEVICE)       # (N_l, 3)
        lig_true = lig_true.squeeze(0).to(DEVICE)         # (N_l, 3)
        lig_feats = lig_feats.squeeze(0).to(DEVICE)       # (N_l, 10)

        optimizer.zero_grad()

        # 前向传播：从加噪坐标出发，预测去噪后的坐标
        lig_pred = model(prot_feats, prot_coords, lig_feats, lig_noisy)

        # 损失：预测坐标与真实坐标的 RMSD
        loss = compute_rmsd(lig_pred, lig_true)

        # 若加噪后配体距蛋白过远导致无交互边，模型输出无梯度，需跳过
        if loss.requires_grad:
            loss.backward()
            # 梯度裁剪——EGNN 坐标更新可能导致梯度爆炸
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
            optimizer.step()

        epoch_losses.append(loss.item())

    avg_loss = np.mean(epoch_losses) if epoch_losses else float('nan')
    train_losses_history.append(avg_loss)

    # ---- 每 20 轮在验证集评估一次 ----
    if epoch % 20 == 0 or epoch == 1:
        model.eval()
        val_rmsds = []
        with torch.no_grad():
            for prot_coords, prot_feats, lig_noisy, lig_true, lig_feats in test_loader:
                prot_coords = prot_coords.squeeze(0).to(DEVICE)
                prot_feats = prot_feats.squeeze(0).to(DEVICE)
                lig_noisy = lig_noisy.squeeze(0).to(DEVICE)
                lig_true = lig_true.squeeze(0).to(DEVICE)
                lig_feats = lig_feats.squeeze(0).to(DEVICE)

                lig_pred = model(prot_feats, prot_coords, lig_feats, lig_noisy)
                rmsd = compute_rmsd(lig_pred, lig_true).item()
                val_rmsds.append(rmsd)

        avg_val = np.mean(val_rmsds) if val_rmsds else float('nan')
        print(f"  Epoch {epoch:>4d}/{N_EPOCHS}  |  "
              f"Train RMSD: {avg_loss:.4f} A  |  Val RMSD: {avg_val:.4f} A")

## 6. 评估与可视化

评估对接模型的核心指标是 **RMSD (Root Mean Square Deviation)**：

- **RMSD < 2 A**：通常认为是成功的对接预测
- **RMSD < 5 A**：配体大致位于正确的结合口袋内

我们对比两组 RMSD：
1. **加噪后 RMSD**（基线）：不做任何预测的误差，反映噪声大小
2. **预测后 RMSD**（模型输出）：模型去噪后的误差，反映去噪效果

In [ ]:
# ============================================================
# 测试集评估
# ============================================================

# 固定随机种子以确保测试结果可复现
np.random.seed(SEED + 1)

model.eval()
test_rmsds_before = []   # 加噪后（去噪前）的 RMSD —— 衡量噪声大小
test_rmsds_after = []    # 模型预测后的 RMSD —— 衡量去噪效果

with torch.no_grad():
    for prot_coords, prot_feats, lig_noisy, lig_true, lig_feats in test_loader:
        prot_coords = prot_coords.squeeze(0).to(DEVICE)
        prot_feats = prot_feats.squeeze(0).to(DEVICE)
        lig_noisy = lig_noisy.squeeze(0).to(DEVICE)
        lig_true = lig_true.squeeze(0).to(DEVICE)
        lig_feats = lig_feats.squeeze(0).to(DEVICE)

        # 加噪后的 RMSD（基线：不做任何预测的误差）
        rmsd_before = compute_rmsd(lig_noisy, lig_true).item()
        test_rmsds_before.append(rmsd_before)

        # 模型预测后的 RMSD（去噪效果）
        lig_pred = model(prot_feats, prot_coords, lig_feats, lig_noisy)
        rmsd_after = compute_rmsd(lig_pred, lig_true).item()
        test_rmsds_after.append(rmsd_after)

test_rmsds_before = np.array(test_rmsds_before)
test_rmsds_after = np.array(test_rmsds_after)

# ---- 统计指标 ----
mean_before = test_rmsds_before.mean()
mean_after = test_rmsds_after.mean()
median_after = np.median(test_rmsds_after)
lt2 = (test_rmsds_after < 2.0).mean() * 100    # RMSD < 2A 的比例
lt5 = (test_rmsds_after < 5.0).mean() * 100    # RMSD < 5A 的比例

# 用 DataFrame 展示结果
results_df = pd.DataFrame({
    '指标': ['测试样本数', '加噪 RMSD 均值', '预测 RMSD 均值', '预测 RMSD 中位数',
             'RMSD < 2A 比例', 'RMSD < 5A 比例'],
    '值': [f'{len(test_rmsds_after)}',
           f'{mean_before:.2f} A',
           f'{mean_after:.2f} A',
           f'{median_after:.2f} A',
           f'{lt2:.1f}%',
           f'{lt5:.1f}%']
})
print("测试集评估结果：")
display(results_df)

In [ ]:
# ============================================================
# 可视化：RMSD 直方图 + 训练曲线
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左图: RMSD 直方图——对比加噪前后
ax = axes[0]
upper = max(test_rmsds_before.max(), test_rmsds_after.max()) + 1
bins = np.linspace(0, upper, 30)
ax.hist(test_rmsds_before, bins=bins, alpha=0.5,
        label=f'Before (noised), mean={mean_before:.1f}A',
        color='salmon', edgecolor='darkred', linewidth=0.5)
ax.hist(test_rmsds_after, bins=bins, alpha=0.5,
        label=f'After (predicted), mean={mean_after:.1f}A',
        color='steelblue', edgecolor='darkblue', linewidth=0.5)
ax.axvline(x=2.0, color='green', linestyle='--', linewidth=1.0, label='2A threshold')
ax.axvline(x=5.0, color='orange', linestyle='--', linewidth=1.0, label='5A threshold')
ax.set_xlabel('RMSD (A)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Ligand RMSD Distribution', fontsize=13)
ax.legend(fontsize=9)

# 右图: 训练损失曲线
ax = axes[1]
ax.plot(range(1, len(train_losses_history) + 1), train_losses_history,
        color='steelblue', linewidth=1.5)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Training RMSD (A)', fontsize=12)
ax.set_title('Training Loss Curve', fontsize=13)
ax.grid(True, alpha=0.3)

plt.suptitle('KarmaDock Toy Model (EGNN)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 总结

本教程展示了 KarmaDock 的核心思想——使用 **E(n)-等变图神经网络 (EGNN)** 进行分子对接。

### 关键要点回顾

1. **E(n)-等变性**：EGNN 的坐标更新只依赖原子间方向向量和标量权重，对输入施加任意旋转/平移后输出做相同变换
2. **加噪-去噪训练**：对配体坐标加随机噪声（旋转+平移），训练模型恢复正确构象
3. **Recycle 机制**：多次循环精炼，每轮重新建边、重新传递，实现"粗调 -> 细调"
4. **蛋白刚体约束**：坐标更新仅作用于配体原子，蛋白坐标保持固定（半柔性对接）

### 与真实 KarmaDock 的差异

| 方面 | 本教程（玩具模型） | 真实 KarmaDock |
|------|---------------------|----------------|
| 原子特征 | 10 维 (元素+芳香性) | 更丰富的化学特征 |
| 图结构 | 仅蛋白-配体交互边 | 包含内部键、多尺度边 |
| EGNN | 基础版本 | 带坐标归一化、多头注意力 |
| 训练数据 | 285 个复合物 | PDBbind 全集 (>19000) |
| 置信度 | 无 | 模块化置信度估计 |

### 延伸阅读

- **EGNN**: Satorras et al., *E(n) Equivariant Graph Neural Networks*, ICML 2021
- **AlphaFold2 Recycling**: Jumper et al., *Highly accurate protein structure prediction with AlphaFold*, Nature 2021
- **KarmaDock**: Zhang et al., *KarmaDock: a deep learning paradigm for ultra-large library docking with fast modular confidence estimation*, 2023